In [2]:
import os

from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-20b")
model.invoke("what is ai")

AIMessage(content='**Artificial Intelligence (AI)** is a branch of computer science that focuses on building systems capable of performing tasks that normally require human intelligence. These tasks include:\n\n| Domain | Typical AI Tasks |\n|--------|------------------|\n| **Reasoning & Decision‑Making** | Planning, problem solving, inference engines |\n| **Learning** | Machine learning, deep learning, reinforcement learning |\n| **Perception** | Computer vision, speech recognition, natural language processing |\n| **Interaction** | Dialogue systems, chatbots, virtual assistants |\n\n### Core Ideas\n1. **Representation** – How knowledge is encoded (rules, neural networks, symbolic logic).  \n2. **Algorithms** – Procedures that manipulate representations to achieve goals.  \n3. **Learning** – Adjusting representations from data, often using statistical methods.  \n4. **Autonomy** – Acting without continuous human oversight.\n\n### Key Subfields\n- **Machine Learning** – Algorithms that

In [4]:
#csv files

import pandas as pd

data=pd.DataFrame(
    {
    "Product":["Laptop","Mobile", "Tablet", "Laptop", "Mobile", "Tablet"],
    "Sales": [50000, 30000, 20000, 60000, 35000, 25000],
    "Quantity": [2, 3, 1, 3, 4, 2],
    "Region": ["North", "South", "East", "North", "West", "South"]
    }
)

data

,Product,Sales,Quantity,Region
0,Laptop,50000,2,North
1,Mobile,30000,3,South
2,Tablet,20000,1,East
3,Laptop,60000,3,North
4,Mobile,35000,4,West
5,Tablet,25000,2,South


In [5]:
from langchain.tools import tool

@tool
def calculate_total_sales()->float:
    """Calculate the total sales in the dataset."""

    return  float(data["Sales"].sum())


@tool
def region_wise_sales()->float:
    """Calculate the region-wise sales in the dataset."""

    region_wise=data.groupby("Region")["Sales"].sum()
    return region_wise

@tool
def product_highest_sales()->float:
    """Calculate which product has highest sales in the dataset."""

    product_wise=data.groupby("Product")["Sales"].sum()

    return product_wise




In [6]:
print(calculate_total_sales.invoke({}))
print(region_wise_sales.invoke({}))
print(product_highest_sales.invoke({}))

220000.0
Region
East      20000
North    110000
South     55000
West      35000
Name: Sales, dtype: int64
Product
Laptop    110000
Mobile     65000
Tablet     45000
Name: Sales, dtype: int64


In [7]:
from langchain.agents import create_agent

agent=create_agent(
    model=model,
    tools=[
        calculate_total_sales,region_wise_sales,product_highest_sales
    ],
    system_prompt="""
    You are an AI Data Analyst.

    Answer the user's questions about the sales dataset.
    Use the available tools whenever calculations or
    dataset information are required.

    Explain the results in simple language.
    Do not make up data.
    """
)


In [8]:
from langchain.messages import HumanMessage
total_sales=agent.invoke(
    {
        "messages":[
            HumanMessage("what are the total sales") 
        ]
    }
)

print(total_sales["messages"][-2].content)


220000.0


In [9]:
total_sales["messages"]

[HumanMessage(content='what are the total sales', additional_kwargs={}, response_metadata={}, id='7200cade-bb80-4ae0-aa3b-03729f8ee103'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call function calculate_total_sales.', 'tool_calls': [{'id': 'fc_0bd99ddc-b66d-4445-a75e-0c80c16662c7', 'function': {'arguments': '{}', 'name': 'calculate_total_sales'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 220, 'total_tokens': 250, 'completion_time': 0.033103408, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.010910308, 'prompt_tokens_details': None, 'queue_time': 0.384477732, 'total_time': 0.044013716}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ce54c53bd5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17c15-db04-75d3-bd52-4ea87c694c54-0', tool_calls=[{'name': 'calculate_total_sales', 'args': {},

In [10]:
data.to_csv("sales.csv")

In [12]:
from pydantic import BaseModel,Field

class Data_analysis(BaseModel):

    summary:str=Field(description="short summary of the analysis")
    key_findings:list[str]=Field(description="what are the key finding value in analaysis")
    recommendations: list[str] = Field(description="Suggestions based on the findings")
    totalsales:int=Field(description="what are the total sales")

data_with_pydantic=model.with_structured_output(Data_analysis)

data_with_pydantic.invoke("""
Analyze this sales data:
Laptop sales = 110000
Mobile sales = 65000
Tablet sales = 45000

Calculate total sales and provide key findings and recommendations.
""")


Data_analysis(summary='The total sales amount to 220,000, with laptops contributing half of the revenue, mobile sales making up nearly a third, and tablets the remainder. The data suggests focusing on cross‑selling and bundle deals to enhance tablet and mobile sales.', key_findings=['Laptop sales are the highest at 110,000, representing 50% of total sales.', 'Mobile sales are moderate at 65,000 (29.5% of total).', 'Tablet sales are the lowest at 45,000 (20.5% of total).'], recommendations=['Leverage strong laptop performance to cross-sell tablets and mobiles.', 'Introduce bundle offers to increase tablet sales.', 'Invest in targeted mobile marketing to boost its share.'], totalsales=220000)